## Implementasi

### Autentikasi CDSE

In [1]:
import json
import openeo

In [2]:
conn = openeo.connect('openeo.dataspace.copernicus.eu').authenticate_oidc()

Authenticated using refresh token.


### Mengambil AoI di Wilayah Jawa Timur (Banyuwangi-Probolinggo)

In [3]:
BANDS = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']
BBOX = {"west": 113.25, "south": -8.84, "east": 114.64, "north": -7.56, "crs": "EPSG:4326"}

cube = conn.load_collection(
    'SENTINEL2_L2A',
    spatial_extent=BBOX,
    temporal_extent=['2026-06-01', '2026-09-30'],
    bands=BANDS + ['SCL'],
    max_cloud_cover=40,
)

In [4]:
# Buang awan, bayangan awan, cirrus, salju, saturated, no data (berdasarkan SCL)
scl = cube.band('SCL')
bad = (scl == 0) | (scl == 1) | (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)
cube = cube.mask(bad).filter_bands(BANDS)

In [ ]:
cube = cube.reduce_dimension(dimension='t', reducer='median')
cube = cube.resample_spatial(resolution=20, projection=32749)
cube = cube.save_result(format='GTiff')

In [7]:
job = conn.job('j-2610111100034dcbb01faa4d0bcdc745')
print(job.logs(level='ERROR'))

[{'id': '[1791718348232, 351411]', 'time': '2026-10-11T11:32:28.232Z', 'level': 'error', 'message': 'Exception while beginning fetch of 1 outstanding blocks'}, {'id': '[1791718348238, 499041]', 'time': '2026-10-11T11:32:28.238Z', 'level': 'error', 'message': 'Failed to get block(s) from 10.42.176.203:39735'}, {'id': '[1791718348323, 883771]', 'time': '2026-10-11T11:32:28.323Z', 'level': 'error', 'message': 'Stage error: org.apache.spark.shuffle.FetchFailedException\n\tat org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:439)\n\tat org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1253)\n\tat org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:983)\n\tat org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:87)\n\tat org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)\n\tat scala.collection.Itera

In [6]:
# AoI sebesar ini harus lewat batch job (bukan download langsung)
job = cube.create_job(title='s2_aoi_median_20m')
job.start_and_wait()

0:00:00 Job 'j-2610111100034dcbb01faa4d0bcdc745': send 'start'
0:00:03 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:00:08 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:00:15 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:00:23 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:00:33 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:00:46 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:01:02 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:01:21 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:01:45 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:02:15 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:02:53 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:03:40 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:04:39 Job 'j-2610111100034dcbb01faa4d0bcdc745': queued (progress 0%)
0:05:39 Job 'j

JobFailedException: Batch job 'j-2610111100034dcbb01faa4d0bcdc745' didn't finish successfully. Status: error (after 0:38:56).

In [ ]:
job.get_results().download_files('openeo_out')

In [ ]:
import glob
import rasterio

for p in glob.glob('openeo_out/*.tif*'):
    with rasterio.open(p) as src:
        print(p, src.count, 'band,', src.width, 'x', src.height, src.crs)